In [ ]:
# NORTHSTAR -- Tower 45 Cloud Twin Browser QA for Solace Browser
# DNA: twin(cloud) = spawn(gcloud) x session(persist) x control(remote) x stream(visual) x auth(oauth3) x corporate(bypass) x cost(cap)
# Probes cloud runner, Docker, Kubernetes, session persistence, remote control, OAuth3 auth
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "cloud-twin-t45-qa"
NOTEBOOK_PATH = "notebooks/qa/23-cloud-twin-t45-qa.ipynb"
TOWER = 45
TOWER_NAME = "Tower of Cloud Twin Browser"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
CLOUD = SRC / "cloud_runner"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

In [ ]:
# F1 -- Michael Chen: Corporate Restricted (Fortune 500 Analyst)
# Q: Can I access cloud twin from any browser without installing?

# Probe: cloud_app.py exists with API endpoints
cloud_app = read_file(CLOUD / "cloud_app.py")
has_cloud_api = bool(re.search(r'route|endpoint|api|start|stop|status', cloud_app, re.IGNORECASE))
record("F1-P1", "Cloud app module with API endpoints exists", has_cloud_api,
       f"{len(cloud_app)} bytes")

# Probe: tunnel connect page for browser-based access
tunnel_html = read_file(WEB / "tunnel-connect.html")
has_tunnel = len(tunnel_html) > 200
record("F1-P2", "Tunnel connect page for remote access exists", has_tunnel,
       f"{len(tunnel_html)} bytes")

# Probe: machine tunnel for remote connectivity
machine_tunnel = read_file(SRC / "machine" / "tunnel.py")
has_tunnel_code = bool(re.search(r'tunnel|connect|websocket|remote|stream', machine_tunnel, re.IGNORECASE))
record("F1-P3", "Machine tunnel module for remote connectivity", has_tunnel_code,
       f"{len(machine_tunnel)} bytes")

# Probe: sync client for cloud session sync
sync_code = read_file(SRC / "sync_client.py")
has_sync = bool(re.search(r'sync|upload|download|cloud|remote', sync_code, re.IGNORECASE))
record("F1-P4", "Sync client for cloud session data exists", has_sync,
       f"{len(sync_code)} bytes")

In [ ]:
# F2 -- Sarah Kim: Twin Provisioning (Product Manager)
# Q: Does twin run in isolated container? Docker/Kubernetes?

# Probe: Docker configuration exists
docker_dir = CLOUD / "docker"
docker_files = list(docker_dir.rglob("*")) if docker_dir.exists() else []
dockerfile = [f for f in docker_files if 'dockerfile' in f.name.lower() or f.name == 'Dockerfile']
record("F2-P1", "Docker configuration exists for cloud twin",
       len(dockerfile) > 0 or len(docker_files) > 0,
       f"{len(docker_files)} files in docker/, dockerfiles={len(dockerfile)}")

# Probe: Kubernetes configuration exists
k8s_dir = CLOUD / "kubernetes"
k8s_files = list(k8s_dir.rglob("*")) if k8s_dir.exists() else []
record("F2-P2", "Kubernetes configuration exists",
       len(k8s_files) > 0, f"{len(k8s_files)} k8s config files")

# Probe: cloud __init__.py exports
cloud_init = read_file(CLOUD / "__init__.py")
record("F2-P3", "Cloud runner package initialized",
       len(cloud_init) >= 0, f"{len(cloud_init)} bytes")

In [ ]:
# F3 -- Remote Control + Visual Stream
# Q: Can I see live visual stream of the twin browser?

# Probe: machine API exposes control endpoints
machine_api = read_file(SRC / "machine" / "api.py")
has_control = bool(re.search(r'control|command|action|navigate|screenshot', machine_api, re.IGNORECASE))
record("F3-P1", "Machine API exposes remote control endpoints", has_control,
       f"{len(machine_api)} bytes")

# Probe: machine file browser for remote file access
file_browser = read_file(SRC / "machine" / "file_browser.py")
has_files = bool(re.search(r'file|browse|list|read|download', file_browser, re.IGNORECASE))
record("F3-P2", "Machine file browser for remote access exists", has_files,
       f"{len(file_browser)} bytes")

# Probe: machine terminal for remote command execution
terminal = read_file(SRC / "machine" / "terminal.py")
has_terminal = bool(re.search(r'terminal|command|exec|shell|run', terminal, re.IGNORECASE))
record("F3-P3", "Machine terminal for remote commands exists", has_terminal,
       f"{len(terminal)} bytes")

# Probe: machine dashboard for monitoring
dashboard = read_file(WEB / "machine-dashboard.html")
has_dashboard = len(dashboard) > 200
record("F3-P4", "Machine dashboard page exists", has_dashboard,
       f"{len(dashboard)} bytes")

In [ ]:
# F4 -- OAuth3 Auth for Twin + F5 -- Session Persistence
# Q: Does twin use same OAuth3 scoped consent?
# Q: Does session persist if browser tab closes?

# Probe: session manager handles cloud twin sessions
session_code = read_file(SRC / "session_manager.py")
has_persist = bool(re.search(r'persist|save|restore|resume|timeout|ttl', session_code, re.IGNORECASE))
record("F4-P1", "Session manager supports persistence", has_persist,
       f"{len(session_code)} bytes")

# Probe: OAuth3 enforcement protects twin access
enforcement = read_file(SRC / "oauth3" / "enforcement.py")
has_enforce = bool(re.search(r'enforce|check|require|scope|token', enforcement, re.IGNORECASE))
record("F4-P2", "OAuth3 enforcement protects cloud twin access", has_enforce)

# Probe: cloud execution tests exist
cloud_tests = list(TESTS.glob("test_cloud_*.py"))
record("F5-P1", "Cloud execution tests exist", len(cloud_tests) >= 1,
       f"{len(cloud_tests)} cloud test files: {[t.name for t in cloud_tests]}")

# Probe: cloud app endpoints test
cloud_endpoint_test = TESTS / "test_cloud_app_endpoints.py"
has_endpoint_test = cloud_endpoint_test.exists() and cloud_endpoint_test.stat().st_size > 100
record("F5-P2", "Cloud app endpoints test exists", has_endpoint_test)

In [ ]:
# F6 -- Cost Capping + F7 -- Corporate Bypass
# Q: Is there clear cost indicator per minute?

# Probe: budget gates can cap cloud twin costs
budget = read_file(SRC / "budget_gates.py")
has_cap = bool(re.search(r'cap|max|limit|budget|cost', budget, re.IGNORECASE))
record("F6-P1", "Budget gates can cap cloud twin costs", has_cap)

# Probe: auto-update module for keeping twin current
auto_update = read_file(SRC / "auto_update.py")
has_update = bool(re.search(r'update|version|download|upgrade', auto_update, re.IGNORECASE))
record("F6-P2", "Auto-update module exists", has_update,
       f"{len(auto_update)} bytes")

# Probe: machine scopes limit remote access
machine_scopes = read_file(SRC / "machine" / "scopes.py")
has_machine_scopes = bool(re.search(r'scope|permission|allow|deny', machine_scopes, re.IGNORECASE))
record("F6-P3", "Machine scopes limit remote access", has_machine_scopes,
       f"{len(machine_scopes)} bytes")

In [ ]:
# NEGATIVE SPACE (P16) -- Cloud twin gaps

# Probe: tunnel test exists
tunnel_test = TESTS / "test_tunnel_client.py"
has_tunnel_test = tunnel_test.exists() and tunnel_test.stat().st_size > 100
record("NS-P1", "Tunnel client test exists", has_tunnel_test)

# Probe: machine access test
machine_access_test = TESTS / "test_machine_access.py"
has_access_test = machine_access_test.exists() and machine_access_test.stat().st_size > 100
record("NS-P2", "Machine access test exists", has_access_test)

# Probe: no bare excepts in cloud runner (Fallback Ban)
cloud_files = list(CLOUD.glob("*.py")) if CLOUD.exists() else []
bare_excepts = 0
for cf in cloud_files:
    content = cf.read_text(encoding='utf-8', errors='replace')
    bare_excepts += len(re.findall(r'^\s*except\s*:', content, re.MULTILINE))
    bare_excepts += len(re.findall(r'^\s*except\s+Exception\s*:', content, re.MULTILINE))
record("NS-P3", "No bare excepts in cloud runner (Fallback Ban)",
       bare_excepts == 0, f"{bare_excepts} bare excepts across {len(cloud_files)} files")

# Probe: cloud concurrent test
concurrent_test = TESTS / "test_cloud_concurrent.py"
has_conc_test = concurrent_test.exists() and concurrent_test.stat().st_size > 100
record("NS-P4", "Cloud concurrency test exists", has_conc_test)

In [ ]:
# EVIDENCE SUMMARY -- Tower 45 Cloud Twin Browser QA
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"

summary = {
    "surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total, "passed": passed, "failed": failed,
    "score": score, "grade": grade, "finding_rate": finding_rate,
    "converged": finding_rate < 5,
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}

print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))